In [ ]:
"""
实验二：双平台静态策略实验
"""

import os
import re
import json
import random
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

# ================= 配置区 =================
MODEL_PATH = "Qwen/Qwen2.5-14B-Instruct"
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/eco-simp"
MEMORY_DIR = os.path.join(BASE_DIR, "exp2_memories_v2_1")
RESULTS_DIR = os.path.join(BASE_DIR, "exp2_results_v2_1")
MAX_MEMORY_LINES = 30
NUM_ROUNDS = 5
EXIT_LOSE_ROUNDS = 2
COOLDOWN_ROUNDS = 2
TEST_ONLY = False

# 全局随机种子
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# A(内容电商) 佣金高/低, 补贴高/低
A_COMM_HIGH, A_COMM_LOW = 0.14, 0.06
A_SUB_HIGH, A_SUB_LOW = 0.05, 0.01
# B(货架电商) 佣金高/低, 补贴高/低
B_COMM_HIGH, B_COMM_LOW = 0.21, 0.12
B_SUB_HIGH, B_SUB_LOW = 0.08, 0.04

# 16组策略组合
ALL_COMBOS = []
for a_comm in [A_COMM_LOW, A_COMM_HIGH]:
    for a_sub in [A_SUB_LOW, A_SUB_HIGH]:
        for b_comm in [B_COMM_LOW, B_COMM_HIGH]:
            for b_sub in [B_SUB_LOW, B_SUB_HIGH]:
                ALL_COMBOS.append({
                    "A_comm": a_comm, "A_sub": a_sub,
                    "B_comm": b_comm, "B_sub": b_sub
                })

if TEST_ONLY:
    # 只跑第1组和最后1组（差异最大）
    STRATEGY_COMBOS = [ALL_COMBOS[0], ALL_COMBOS[-1]]
else:
    # 跳过已跑的2组，只跑中间14组
    STRATEGY_COMBOS = ALL_COMBOS[1:-1]

os.makedirs(RESULTS_DIR, exist_ok=True)

# ================= 1. 模型加载 =================
print(f"加载模型: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)
print("模型加载完成")


def generate_llm_response(prompt, max_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens,
            do_sample=True, temperature=0.75, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return resp.strip()


# ================= 2. Agent 基类 =================
class BaseAgent:
    def __init__(self, agent_id, role_type, desc):
        self.agent_id = agent_id
        self.role_type = role_type
        self.description = desc
        self.memory_file = ""

    def init_memory(self):
        with open(self.memory_file, "w", encoding="utf-8") as f:
            f.write(f"【角色】{self.role_type}｜{self.description}\n【历史】\n")

    def read_memory(self):
        if not os.path.exists(self.memory_file):
            return ""
        with open(self.memory_file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        if len(lines) <= MAX_MEMORY_LINES + 1:
            return "".join(lines)
        return "".join(lines[:1] + lines[-MAX_MEMORY_LINES:])

    def append_memory(self, text):
        with open(self.memory_file, "a", encoding="utf-8") as f:
            f.write(text + "\n")

    def decide(self, context, task):
        memory = self.read_memory()
        prompt = self.build_prompt(memory, context, task)
        res = generate_llm_response(prompt)
        think_match = re.search(r"<thinking>(.*?)</thinking>", res, re.DOTALL | re.I)
        dec_match = re.search(r"<decision>(.*?)</decision>", res, re.DOTALL | re.I)
        thinking = think_match.group(1).strip() if think_match else ""
        decision = dec_match.group(1).strip() if dec_match else res.strip()
        self.append_memory(f"情境：{context[:80]}...｜决策：{decision}")
        return decision, thinking

    def build_prompt(self, memory, context, task):
        return ""


# ================= 3. 商家Agent =================
SELLER_CONFIGS = [
    ("Seller_L1", "低价走量", 30, 45, 55, 0.00, 3, "保守求生，倾向低价低广告，追求稳定出单"),
    ("Seller_L2", "低价走量", 30, 45, 55, 0.00, 3, "薄利多销，愿意在定价区间中段定价换取更多销量"),
    ("Seller_L3", "低价走量", 30, 45, 55, 0.00, 3, "轻微卷价，喜欢比同行低1-2元抢单"),
    ("Seller_L4", "低价走量", 30, 45, 55, 0.00, 3, "广告厌恶型，几乎不投广告，依靠自然流量"),
    ("Seller_L5", "低价走量", 30, 45, 55, 0.00, 3, "跟随市场，参考上轮结果调整策略"),
    ("Seller_L6", "低价走量", 30, 45, 55, 0.00, 3, "激进投放型，愿意在低价品上投满广告博曝光"),
    ("Seller_M1", "中端品质", 50, 130, 170, 0.05, 15, "均衡稳健，追求利润与销量的平衡点"),
    ("Seller_M2", "中端品质", 50, 130, 170, 0.05, 15, "品牌导向，倾向高定价维护品牌调性，广告适度"),
    ("Seller_M3", "中端品质", 50, 130, 170, 0.05, 15, "转化优先，愿意降价提升转化率，广告投入较高"),
    ("Seller_H1", "高端溢价", 100, 270, 330, 0.10, 45, "高端长期主义，坚持高定价，重品牌形象和复购"),
]


class SellerAgent(BaseAgent):
    def __init__(self, sid, seller_type, cost, price_min, price_max, margin_safe, ad_max, character):
        super().__init__(sid, seller_type, f"{seller_type}｜{character}")
        self.seller_type = seller_type
        self.unit_cost = cost
        self.price_min = price_min
        self.price_max = price_max
        self.margin_safe = margin_safe
        self.ad_max = ad_max
        self.character = character
        # 双平台状态
        self.last_margin_A = 0.0
        self.last_margin_B = 0.0
        self.last_sales_A = 0
        self.last_sales_B = 0
        self.consecutive_lose_A = 0
        self.consecutive_lose_B = 0
        self.exited_A = False
        self.exited_B = False
        self.cooldown_A = 0
        self.cooldown_B = 0

    def reset(self):
        self.last_margin_A = 0.0
        self.last_margin_B = 0.0
        self.last_sales_A = 0
        self.last_sales_B = 0
        self.consecutive_lose_A = 0
        self.consecutive_lose_B = 0
        self.exited_A = False
        self.exited_B = False
        self.cooldown_A = 0
        self.cooldown_B = 0

    def build_prompt(self, memory, context, task):
        margin_pct = f"{self.margin_safe:.0%}" if self.margin_safe > 0 else "0%（可微利生存）"
        return f"""你是一个同时在两个电商平台（A和B）经营的商家，需要根据双平台政策和市场环境做出理性的经营决策。

【商家身份】
- 类型：{self.seller_type}
- 经营性格：{self.character}
- 单位生产成本：￥{self.unit_cost}
- 允许定价区间：￥{self.price_min} ~ ￥{self.price_max}（两平台同价，必须在此区间内）
- 每个平台广告投入上限：￥{self.ad_max}（0到{self.ad_max}之间）
- 毛利安全线：{margin_pct}

【双平台差异】
- 平台A（内容电商）：依赖达人推荐和内容种草驱动销售，广告曝光机制为前2名高曝光、3-4名中曝光、5-6名低曝光。佣金率通常较低。
- 平台B（货架电商）：用户主动搜索购买，广告竞价更集中（前1名高曝光、2-3名中曝光、4-6名低曝光）。佣金率通常较高但用户购买意图明确。

【经营常识提醒】
- 每个平台的利润 = 该平台销售收入 - 生产成本 - 平台佣金 - 该平台广告费 - 达人佣金（仅A平台有达人）
- 广告费无论是否卖出都要支付，请谨慎投放
- 你需要在两个平台之间合理分配广告预算
- 佣金率越高，利润空间越小；补贴率越高，用户实付更低，理论上有利于销量
- 请根据你的性格特点、两平台各自的上轮表现，做出差异化策略

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先进行策略分析（考虑两平台的成本差异、竞争环境、你的性格倾向），然后给出具体决策。
严格按格式输出，价格必须在{self.price_min}到{self.price_max}之间，广告必须在0到{self.ad_max}之间：
<thinking>你的策略分析（2-3句话）</thinking>
<decision>价格:X,广告A:Y,广告B:Z</decision>"""

    def decide_action(self, polA, polB):
        """商家在双平台环境下的决策：定价 + AB平台各自广告投入"""
        # 构建上轮状态描述
        if self.exited_A:
            status_A = f"已退出平台A（冷却剩余{self.cooldown_A}轮）"
        elif self.last_sales_A > 0:
            status_A = f"毛利率{self.last_margin_A:.1%}（{'盈利' if self.last_margin_A > self.margin_safe else '低于安全线'}），销量{self.last_sales_A}件，连续亏损{self.consecutive_lose_A}轮"
        else:
            status_A = f"上轮无销量，连续亏损{self.consecutive_lose_A}轮"

        if self.exited_B:
            status_B = f"已退出平台B（冷却剩余{self.cooldown_B}轮）"
        elif self.last_sales_B > 0:
            status_B = f"毛利率{self.last_margin_B:.1%}（{'盈利' if self.last_margin_B > self.margin_safe else '低于安全线'}），销量{self.last_sales_B}件，连续亏损{self.consecutive_lose_B}轮"
        else:
            status_B = f"上轮无销量，连续亏损{self.consecutive_lose_B}轮"

        ctx = (f"平台A（内容电商）：佣金率={polA['commission']:.0%}，补贴率={polA['subsidy']:.0%}（用户实付打{1-polA['subsidy']:.0%}折）\n"
               f"  你在A的状态：{status_A}\n"
               f"平台B（货架电商）：佣金率={polB['commission']:.0%}，补贴率={polB['subsidy']:.0%}（用户实付打{1-polB['subsidy']:.0%}折）\n"
               f"  你在B的状态：{status_B}\n"
               f"同类竞品约{sum(1 for c in SELLER_CONFIGS if c[1]==self.seller_type)}家在两平台售卖。\n"
               f"连续{EXIT_LOSE_ROUNDS}轮低于安全线将被迫退出该平台。")

        active_on = []
        if not self.exited_A:
            active_on.append("A")
        if not self.exited_B:
            active_on.append("B")

        if not active_on:
            return {"price": 0, "ad_A": 0, "ad_B": 0, "thinking": "两平台均已退出"}

        task = (f"你当前在平台{'、'.join(active_on)}经营。请决定：\n"
                f"1. 商品定价（￥{self.price_min}~￥{self.price_max}，两平台同价）\n"
                f"2. 平台A广告投入（￥0~￥{self.ad_max}）{'——已退出，必须填0' if self.exited_A else ''}\n"
                f"3. 平台B广告投入（￥0~￥{self.ad_max}）{'——已退出，必须填0' if self.exited_B else ''}\n"
                f"注意：广告费无论是否卖出都会扣除。请考虑佣金率差异（A={polA['commission']:.0%} vs B={polB['commission']:.0%}）来分配投入。")

        dec, thinking = self.decide(ctx, task)

        # 解析价格（默认中位数而非下限）
        price = (self.price_min + self.price_max) / 2
        price_m = re.findall(r'价格[:：]?\s*(\d+\.?\d*)', dec)
        if price_m:
            val = float(price_m[0])
            price = max(self.price_min, min(self.price_max, val))
        else:
            nums = re.findall(r'\d+\.?\d*', dec)
            for n in nums:
                val = float(n)
                if self.price_min <= val <= self.price_max:
                    price = val
                    break

        # 解析广告A
        ad_A = 0.0
        adA_m = re.findall(r'广告A[:：]?\s*(\d+\.?\d*)', dec)
        if adA_m and not self.exited_A:
            val = float(adA_m[0])
            ad_A = max(0, min(self.ad_max, val))

        # 解析广告B
        ad_B = 0.0
        adB_m = re.findall(r'广告B[:：]?\s*(\d+\.?\d*)', dec)
        if adB_m and not self.exited_B:
            val = float(adB_m[0])
            ad_B = max(0, min(self.ad_max, val))

        # 退出的平台广告强制为0
        if self.exited_A:
            ad_A = 0
        if self.exited_B:
            ad_B = 0

        return {"price": round(price, 1), "ad_A": round(ad_A, 1), "ad_B": round(ad_B, 1), "thinking": thinking}


# ================= 4. 达人Agent =================
# 影响力写死，确保可复现
INFLUENCER_CONFIGS = [
    ("Infu_T1", "种草型", 3, 0.06, "温和走心，重视商品品质与粉丝信任，偏好推荐中高端好物", "中高端"),
    ("Infu_T2", "种草型", 2, 0.06, "性价比种草达人，善于发现高性价比商品，均衡务实", "中端"),
    ("Infu_D1", "带货型", 4, 0.04, "冲量带货王，追求单场最大销量，偏好低价爆品走量", "低价"),
    ("Infu_D2", "带货型", 3, 0.04, "稳健带货，追求稳定的佣金收益，不冒进", "中端"),
    ("Infu_D3", "带货型", 5, 0.04, "头部达人，影响力大，偏好高客单价商品以获得更高佣金总额", "中高端"),
]


class InfluencerAgent(BaseAgent):
    def __init__(self, iid, flavor, inf_power, comm_rate, personality, preference):
        super().__init__(iid, "达人", f"{flavor}｜{personality}")
        self.flavor = flavor
        self.inf_power = inf_power
        self.comm_rate = comm_rate
        self.personality = personality
        self.preference = preference
        self.chosen_platform = "A"
        self.chosen_seller = None

    def reset(self):
        self.chosen_platform = "A"
        self.chosen_seller = None

    def build_prompt(self, memory, context, task):
        return f"""你是一个电商带货达人，需要在两个平台之间选择带货平台和合作商家。

【达人身份】
- 达人类型：{self.flavor}
- 性格特点：{self.personality}
- 带货影响力：{self.inf_power}/5（影响力越高，能为商家带来越多用户关注）
- 佣金比例：{self.comm_rate:.0%}（你从带货销售额中抽取的比例）
- 选品偏好：偏好{self.preference}商品

【双平台差异】
- 平台A（内容电商）：达人生态更成熟，内容种草+达人推荐是主要购买驱动力
- 平台B（货架电商）：用户以搜索购买为主，达人推荐的影响力相对较弱，但用户购买意图更明确

【选择逻辑提醒】
- 你的佣金收入 = 你带来的销售额 × {self.comm_rate:.0%}
- 平台A达人影响力更大但竞争可能更激烈；平台B达人影响力较弱但可能有差异化优势
- 高价商品单件佣金高，但转化可能低；低价商品单件佣金低，但走量大
- 请根据你的选品偏好、两平台政策差异来选择

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先分析两个平台的商家情况和政策差异，结合你的偏好做出选择。
<thinking>你的分析（2-3句话）</thinking>
<decision>平台:A/B,商家:商家ID</decision>"""

    def choose_platform_and_seller(self, polA, polB, sellers_A_info, sellers_B_info):
        ctx = (f"平台A（内容电商）：佣金率={polA['commission']:.0%}，补贴率={polA['subsidy']:.0%}\n"
               f"平台A活跃商家：\n{sellers_A_info}\n\n"
               f"平台B（货架电商）：佣金率={polB['commission']:.0%}，补贴率={polB['subsidy']:.0%}\n"
               f"平台B活跃商家：\n{sellers_B_info}")
        task = (f"请选择：1.带货平台（A或B）；2.合作商家ID。你偏好{self.preference}类型的商品。\n"
                f"格式：<decision>平台:A/B,商家:商家ID</decision>")
        dec, thinking = self.decide(ctx, task)

        # 解析平台
        plat_match = re.search(r'平台[:：]?\s*([AB])', dec)
        if plat_match:
            self.chosen_platform = plat_match.group(1)
        elif "B" in dec and "A" not in dec:
            self.chosen_platform = "B"
        else:
            self.chosen_platform = "A"

        # 解析商家
        self.chosen_seller = None
        sids = re.findall(r'Seller_[A-Z]\d+', dec)
        if sids:
            self.chosen_seller = sids[0]

        return self.chosen_platform, self.chosen_seller, thinking


# ================= 5. 用户Agent =================
USER_CONFIGS = []
_low_personas = [
    "极度价格敏感，只买最便宜的商品，哪个平台便宜去哪个",
    "折扣猎手，哪个平台补贴高就去哪个平台",
    "容易被广告吸引，看到高曝光商品就想买",
    "信任达人推荐，达人说好就买，偏好平台A",
    "习惯使用平台A，不太愿意切换平台",
    "有品质意识但预算紧张，在预算内选最好的",
    "冲动型消费者，看到低价好物就下单",
    "理性比价型，会对比两个平台所有低价商品再决定",
    "回头客，倾向于回购之前买过的商家和平台",
    "新用户，好奇心强，两个平台都愿意尝试",
    "促销敏感型，补贴力度大的平台多买",
    "口碑导向，看重达人推荐和用户评价",
    "习惯购买型，偏好固定平台和商家",
    "社交推荐驱动，受达人影响大，偏向内容电商A",
    "随机浏览型，没有强偏好，两平台随机看看",
]
for i in range(15):
    USER_CONFIGS.append((f"User_L{i+1}", "低价敏感", 100, _low_personas[i]))

_mid_personas = [
    "重视达人推荐，达人背书的中端商品优先，偏好平台A",
    "品质优先，愿意为更好品质多付一些，两平台比较",
    "价格适中即可，不追求最低也不追求最贵",
    "理性消费者，几乎不看广告，自主判断，偏好搜索购物的平台B",
    "品牌忠诚型，认准的商家会持续复购",
    "尝鲜型，喜欢试新品和新商家，两平台都逛",
    "精打细算型，追求品质的同时关注性价比",
    "口碑驱动，看重达人和其他用户的推荐",
    "活动敏感，补贴活动期间倾向多买",
    "均衡考量型，综合考虑价格、品质、推荐、平台体验",
]
for i in range(10):
    USER_CONFIGS.append((f"User_M{i+1}", "品质性价比", 200, _mid_personas[i]))

_high_personas = [
    "只买高端精品，价格不是首要考虑因素，两平台都看",
    "深度信任头部达人推荐的高端商品，偏好平台A",
    "纯品质导向，只关注产品本身的品质，偏好搜索比较",
    "品牌忠诚度极高，只买认可的高端商家",
    "自主决策型，不受广告和达人影响，独立判断",
]
for i in range(5):
    USER_CONFIGS.append((f"User_H{i+1}", "高端品质", 400, _high_personas[i]))


class UserAgent(BaseAgent):
    def __init__(self, uid, user_type, budget, persona):
        super().__init__(uid, "用户", f"{user_type}｜{persona}")
        self.user_type = user_type
        self.budget = budget
        self.persona = persona

    def build_prompt(self, memory, context, task):
        return ""

    def build_purchase_prompt(self, memory, context, task, influenced_by_inf, influenced_by_ad):
        """构建双平台购买决策的完整prompt"""
        if self.user_type == "低价敏感":
            type_guide = (
                "作为低价敏感用户，你优先选择最便宜的商品，不管在哪个平台。"
                "中端和高端商品通常超出你的预算，除非补贴后价格在你预算内，否则不要购买。"
                "哪个平台补贴力度大、商品便宜，你就倾向于在那个平台购买。"
            )
        elif self.user_type == "品质性价比":
            type_guide = (
                "作为品质性价比用户，你追求品质与价格的平衡。"
                "你倾向于购买中端商品，低价商品品质可能不够，高端商品超出你的预算考量。"
                "你会在两个平台之间比较，选择性价比最高的商品。"
            )
        else:
            type_guide = (
                "作为高端品质用户，你追求最好的产品品质。"
                "你倾向于购买高端或中高端商品，低价商品基本不在你的考虑范围内。"
                "价格不是你的首要顾虑，品质和品牌才是。你会在两个平台比较同款商品的展示和服务。"
            )

        inf_desc = "本轮你看到了达人推荐内容，达人推荐会影响你的判断（达人主要活跃在平台A）" if influenced_by_inf else "本轮你没有关注达人推荐，更依赖自己的判断"
        ad_desc = "本轮你注意到了广告展示，高曝光商品吸引了你的注意" if influenced_by_ad else "本轮你对广告不敏感，广告曝光不影响你的选择"

        return f"""你是一个可以在两个电商平台（A和B）购物的用户，需要决定在哪个平台买什么商品。

【用户身份】
- 用户类型：{self.user_type}
- 性格特点：{self.persona}
- 本轮预算上限：￥{self.budget}（你绝对不能购买单价超过￥{self.budget}的商品！）

【双平台差异】
- 平台A（内容电商）：有达人推荐和内容种草，购物体验更社交化
- 平台B（货架电商）：搜索购买为主，商品展示更直接，适合明确知道要买什么的用户

【选购指导】
{type_guide}

【本轮状态】
- {inf_desc}
- {ad_desc}

【重要约束】
- 你最多购买2件商品（可来自不同平台），也可以选择不买
- 每件商品的实付价格必须 ≤ 你的预算￥{self.budget}，超过预算的商品禁止购买
- 请注意商品标注的实际支付价格（已含补贴折扣）

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先分析两个平台中哪些商品在你的预算内且符合你的偏好，然后做出决定。
<thinking>你的分析（考虑预算、偏好、平台差异、达人推荐、广告影响）</thinking>
<decision>平台X-商家ID1,平台Y-商家ID2</decision> 或 <decision>不买</decision>
示例：<decision>A-Seller_L1,B-Seller_M2</decision>"""

    def decide_purchase(self, items_A_info, items_B_info, reco_info, influenced_by_inf, influenced_by_ad):
        """浏览AB全部商品后决策，最多买2件，可来自不同平台"""
        memory = self.read_memory()
        ctx = (f"【平台A（内容电商）商品】（价格为补贴后你实际支付的价格）\n{items_A_info}\n\n"
               f"【平台B（货架电商）商品】（价格为补贴后你实际支付的价格）\n{items_B_info}\n\n"
               f"【达人推荐信息】\n{reco_info}")
        task = (f"浏览两个平台后选择购买（最多2件，可来自不同平台，也可不买）。预算￥{self.budget}，不能买超过这个价格的商品。\n"
                f"格式：<decision>平台X-商家ID1,平台Y-商家ID2</decision> 或 <decision>不买</decision>\n"
                f"示例：<decision>A-Seller_L1,B-Seller_M2</decision>")
        prompt = self.build_purchase_prompt(memory, ctx, task, influenced_by_inf, influenced_by_ad)
        res = generate_llm_response(prompt)

        think_match = re.search(r"<thinking>(.*?)</thinking>", res, re.DOTALL | re.I)
        dec_match = re.search(r"<decision>(.*?)</decision>", res, re.DOTALL | re.I)
        thinking = think_match.group(1).strip() if think_match else ""
        decision = dec_match.group(1).strip() if dec_match else res.strip()
        self.append_memory(f"情境：{ctx[:80]}...｜决策：{decision}")

        purchases = []  # list of (platform, seller_id)
        if "不买" in decision or "none" in decision.lower():
            return purchases, thinking

        # 解析 "A-Seller_L1" 或 "B-Seller_M2" 格式
        pairs = re.findall(r'([AB])\s*[-–]\s*(Seller_[A-Z]\d+)', decision)
        if pairs:
            for plat, sid in pairs[:2]:
                purchases.append((plat, sid))
        else:
            # 尝试只提取商家ID，默认平台A
            sids = re.findall(r'Seller_[A-Z]\d+', decision)
            for sid in sids[:2]:
                purchases.append(("A", sid))

        return purchases, thinking


# ================= 6. 广告曝光分配 =================
def assign_exposure_A(seller_ads):
    """内容电商：前2高曝光，3-4中曝光，5-6低曝光"""
    sorted_s = sorted([(s, a) for s, a in seller_ads.items() if a > 0], key=lambda x: x[1], reverse=True)
    exp = {}
    for i, (sid, _) in enumerate(sorted_s):
        if i < 2:
            exp[sid] = "高曝光"
        elif i < 4:
            exp[sid] = "中曝光"
        elif i < 6:
            exp[sid] = "低曝光"
    return exp


def assign_exposure_B(seller_ads):
    """货架电商：前1高曝光，2-3中曝光，4-6低曝光"""
    sorted_s = sorted([(s, a) for s, a in seller_ads.items() if a > 0], key=lambda x: x[1], reverse=True)
    exp = {}
    for i, (sid, _) in enumerate(sorted_s):
        if i < 1:
            exp[sid] = "高曝光"
        elif i < 3:
            exp[sid] = "中曝光"
        elif i < 6:
            exp[sid] = "低曝光"
    return exp


# ================= 7. 构建商品信息字符串 =================
def build_items_info(active_sellers, seller_actions, ad_key, exposure_map, influencer_reco_map, subsidy_rate):
    """构建给用户看的商品列表（展示补贴后价格）"""
    lines = []
    for s in active_sellers:
        sid = s.agent_id
        act = seller_actions[sid]
        if act["price"] == 0:
            continue
        exp = exposure_map.get(sid, "无")
        inf_tag = influencer_reco_map.get(sid, "")
        original_price = act["price"]
        actual_price = round(original_price * (1 - subsidy_rate), 1)
        line = f"  {sid}({s.seller_type}) 实付￥{actual_price}（原价￥{original_price}，补贴{subsidy_rate:.0%}）"
        if exp != "无":
            line += f" 【{exp}】"
        if inf_tag:
            line += f" {inf_tag}"
        lines.append(line)
    return "\n".join(lines) if lines else "  暂无商品"


# ================= 8. 仿真主逻辑 =================
def run_dual_static_experiment(polA, polB, exp_label):
    """运行单组双平台静态实验"""
    print(f"\n{'='*60}")
    print(f"实验组 {exp_label}: A佣{polA['commission']:.2f}补{polA['subsidy']:.2f} | B佣{polB['commission']:.2f}补{polB['subsidy']:.2f}")
    print(f"{'='*60}")

    mem_dir = os.path.join(MEMORY_DIR, exp_label)
    os.makedirs(mem_dir, exist_ok=True)

    # 初始化agents
    sellers = []
    for cfg in SELLER_CONFIGS:
        s = SellerAgent(*cfg)
        s.memory_file = os.path.join(mem_dir, f"{s.agent_id}_memory.txt")
        s.init_memory()
        s.reset()
        sellers.append(s)

    influencers = []
    for cfg in INFLUENCER_CONFIGS:
        inf = InfluencerAgent(*cfg)
        inf.memory_file = os.path.join(mem_dir, f"{inf.agent_id}_memory.txt")
        inf.init_memory()
        inf.reset()
        influencers.append(inf)

    users = []
    for cfg in USER_CONFIGS:
        u = UserAgent(*cfg)
        u.memory_file = os.path.join(mem_dir, f"{u.agent_id}_memory.txt")
        u.init_memory()
        users.append(u)

    round_records = []
    seller_records = []
    influencer_records = []
    user_records = []
    transaction_records = []

    for rnd in range(1, NUM_ROUNDS + 1):
        print(f"\n--- {exp_label} 第{rnd}轮 ---")

        # ---- 冷却处理 ----
        for s in sellers:
            if s.exited_A and s.cooldown_A > 0:
                s.cooldown_A -= 1
                if s.cooldown_A == 0:
                    s.exited_A = False
                    s.consecutive_lose_A = 0
                    print(f"  {s.agent_id} 平台A冷却结束，重新回到平台A")
            if s.exited_B and s.cooldown_B > 0:
                s.cooldown_B -= 1
                if s.cooldown_B == 0:
                    s.exited_B = False
                    s.consecutive_lose_B = 0
                    print(f"  {s.agent_id} 平台B冷却结束，重新回到平台B")

        # ---- 商家决策 ----
        sellers_on_A = [s for s in sellers if not s.exited_A]
        sellers_on_B = [s for s in sellers if not s.exited_B]
        all_active = [s for s in sellers if not s.exited_A or not s.exited_B]
        seller_actions = {}
        for s in all_active:
            act = s.decide_action(polA, polB)
            seller_actions[s.agent_id] = act
            print(f"  {s.agent_id}: 价格={act['price']}, 广告A={act['ad_A']}, 广告B={act['ad_B']}")

        # 全退出的商家
        for s in sellers:
            if s.agent_id not in seller_actions:
                seller_actions[s.agent_id] = {"price": 0, "ad_A": 0, "ad_B": 0, "thinking": "两平台均已退出"}

        # ---- 广告曝光 ----
        ads_A = {s.agent_id: seller_actions[s.agent_id]["ad_A"] for s in sellers_on_A}
        ads_B = {s.agent_id: seller_actions[s.agent_id]["ad_B"] for s in sellers_on_B}
        exposure_A = assign_exposure_A(ads_A)
        exposure_B = assign_exposure_B(ads_B)

        # ---- 达人选平台+商家 ----
        sellers_A_info = "\n".join([
            f"  {s.agent_id}({s.seller_type}) 定价￥{seller_actions[s.agent_id]['price']} 广告投入￥{seller_actions[s.agent_id]['ad_A']}"
            for s in sellers_on_A if seller_actions[s.agent_id]["price"] > 0
        ])
        sellers_B_info = "\n".join([
            f"  {s.agent_id}({s.seller_type}) 定价￥{seller_actions[s.agent_id]['price']} 广告投入￥{seller_actions[s.agent_id]['ad_B']}"
            for s in sellers_on_B if seller_actions[s.agent_id]["price"] > 0
        ])

        influencer_reco_A = {}
        influencer_reco_B = {}
        influencer_chosen = {}  # {inf_id: (platform, seller_id)}
        inf_round_data = []

        for inf in influencers:
            plat, sid, inf_think = inf.choose_platform_and_seller(polA, polB, sellers_A_info, sellers_B_info)
            tag = f"【达人推荐:{inf.agent_id}(影响力{inf.inf_power}/5)】"
            if sid and sid in seller_actions and seller_actions[sid]["price"] > 0:
                if plat == "A":
                    influencer_reco_A[sid] = influencer_reco_A.get(sid, "") + " " + tag
                else:
                    influencer_reco_B[sid] = influencer_reco_B.get(sid, "") + " " + tag
                influencer_chosen[inf.agent_id] = (plat, sid)

            inf_round_data.append({
                "exp": exp_label, "round": rnd,
                "influencer_id": inf.agent_id, "flavor": inf.flavor,
                "inf_power": inf.inf_power, "comm_rate": inf.comm_rate,
                "chosen_platform": plat, "chosen_seller": sid or "无",
                "thinking": inf_think
            })

        # ---- 构建商品展示（展示补贴后价格） ----
        items_A_info = build_items_info(sellers_on_A, seller_actions, "ad_A", exposure_A, influencer_reco_A, polA["subsidy"])
        items_B_info = build_items_info(sellers_on_B, seller_actions, "ad_B", exposure_B, influencer_reco_B, polB["subsidy"])
        all_reco = {}
        for sid, tag in influencer_reco_A.items():
            all_reco[f"A-{sid}"] = tag
        for sid, tag in influencer_reco_B.items():
            all_reco[f"B-{sid}"] = tag
        reco_info = "\n".join([f"  {k}: {v}" for k, v in all_reco.items()]) if all_reco else "  无达人推荐"

        # ---- 构建达人推荐的反向映射 ----
        # {(platform, seller_id): [inf_id, ...]}
        seller_to_influencers = {}
        for inf_id, (plat, sid) in influencer_chosen.items():
            key = (plat, sid)
            seller_to_influencers.setdefault(key, []).append(inf_id)
        inf_recommended_A = {sid for (plat, sid) in influencer_chosen.values() if plat == "A"}
        inf_recommended_B = {sid for (plat, sid) in influencer_chosen.values() if plat == "B"}

        # ---- 用户购买 ----
        round_transactions = []
        user_round_data = []

        for u in users:
            # ★ 统一随机一次
            influenced_by_inf = random.choice([0, 1])
            influenced_by_ad = random.choice([0, 1])

            purchases, u_think = u.decide_purchase(
                items_A_info, items_B_info, reco_info,
                influenced_by_inf, influenced_by_ad
            )

            # ★ 后处理硬过滤——超预算拦截
            valid_purchases = []
            for plat, sid in purchases:
                if sid in seller_actions and seller_actions[sid]["price"] > 0:
                    sub_rate = polA["subsidy"] if plat == "A" else polB["subsidy"]
                    actual_price = round(seller_actions[sid]["price"] * (1 - sub_rate), 1)
                    if actual_price <= u.budget:
                        valid_purchases.append((plat, sid))
                    else:
                        print(f"    ⛔ {u.agent_id} 试图在平台{plat}购买 {sid}(￥{actual_price})超出预算￥{u.budget}，已拦截")

            if not valid_purchases:
                user_round_data.append({
                    "exp": exp_label, "round": rnd,
                    "user_id": u.agent_id, "user_type": u.user_type,
                    "persona": u.persona, "budget": u.budget,
                    "influenced_by_influencer": influenced_by_inf,
                    "influenced_by_ad": influenced_by_ad,
                    "purchased": 0, "purchase_platform": "",
                    "seller_bought": "", "price_paid": 0,
                    "is_influencer_driven": False,
                    "thinking": u_think
                })
            else:
                for plat, sid in valid_purchases:
                    price = seller_actions[sid]["price"]
                    sub_rate = polA["subsidy"] if plat == "A" else polB["subsidy"]
                    actual = round(price * (1 - sub_rate), 1)
                    # 判断是否为达人驱动的订单
                    if plat == "A":
                        is_inf_driven = (influenced_by_inf == 1 and sid in inf_recommended_A)
                    else:
                        is_inf_driven = (influenced_by_inf == 1 and sid in inf_recommended_B)

                    round_transactions.append({
                        "exp": exp_label, "round": rnd,
                        "user_id": u.agent_id, "user_type": u.user_type,
                        "seller_id": sid, "platform": plat,
                        "original_price": price, "subsidy_rate": sub_rate,
                        "actual_price": actual,
                        "is_influencer_driven": is_inf_driven
                    })
                    user_round_data.append({
                        "exp": exp_label, "round": rnd,
                        "user_id": u.agent_id, "user_type": u.user_type,
                        "persona": u.persona, "budget": u.budget,
                        "influenced_by_influencer": influenced_by_inf,
                        "influenced_by_ad": influenced_by_ad,
                        "purchased": 1, "purchase_platform": plat,
                        "seller_bought": sid, "price_paid": actual,
                        "is_influencer_driven": is_inf_driven,
                        "thinking": u_think
                    })

        # ---- 清算 ----
        seller_sales_A = {s.agent_id: 0 for s in sellers}
        seller_sales_B = {s.agent_id: 0 for s in sellers}
        seller_gmv_A = {s.agent_id: 0.0 for s in sellers}
        seller_gmv_B = {s.agent_id: 0.0 for s in sellers}
        seller_inf_driven_gmv_A = {s.agent_id: 0.0 for s in sellers}
        seller_inf_driven_gmv_B = {s.agent_id: 0.0 for s in sellers}

        for tx in round_transactions:
            sid = tx["seller_id"]
            if tx["platform"] == "A":
                seller_sales_A[sid] += 1
                seller_gmv_A[sid] += tx["original_price"]
                if tx["is_influencer_driven"]:
                    seller_inf_driven_gmv_A[sid] += tx["original_price"]
            else:
                seller_sales_B[sid] += 1
                seller_gmv_B[sid] += tx["original_price"]
                if tx["is_influencer_driven"]:
                    seller_inf_driven_gmv_B[sid] += tx["original_price"]

        # ★ 达人佣金：仅基于达人实际带来的订单
        seller_inf_cost_A = {s.agent_id: 0.0 for s in sellers}
        seller_inf_cost_B = {s.agent_id: 0.0 for s in sellers}
        for inf in influencers:
            if inf.agent_id in influencer_chosen:
                plat, sid = influencer_chosen[inf.agent_id]
                num_infs = len(seller_to_influencers.get((plat, sid), []))
                if num_infs > 0:
                    if plat == "A":
                        share_gmv = seller_inf_driven_gmv_A[sid] / num_infs
                        seller_inf_cost_A[sid] += share_gmv * inf.comm_rate
                    else:
                        share_gmv = seller_inf_driven_gmv_B[sid] / num_infs
                        seller_inf_cost_B[sid] += share_gmv * inf.comm_rate

        # ---- 商家清算 ----
        for s in sellers:
            sid = s.agent_id
            act = seller_actions[sid]

            # === 平台A ===
            gmvA = seller_gmv_A[sid]
            salesA = seller_sales_A[sid]
            adA = act["ad_A"]

            if not s.exited_A and sid in [x.agent_id for x in sellers_on_A]:
                commA = gmvA * polA["commission"]
                # ★ 方案A：商家不获得补贴收入，商家收原价GMV
                costA = s.unit_cost * salesA
                infA = seller_inf_cost_A[sid]
                # ★ 广告费始终计入
                profitA = gmvA - costA - commA - adA - infA
                marginA = profitA / gmvA if gmvA > 0 else (-adA if adA > 0 else 0)
                s.last_margin_A = marginA
                s.last_sales_A = salesA

                # 退出判断
                if gmvA == 0 and adA > 0:
                    s.consecutive_lose_A += 1
                elif gmvA > 0 and marginA < s.margin_safe:
                    s.consecutive_lose_A += 1
                elif gmvA > 0:
                    s.consecutive_lose_A = 0

                if s.consecutive_lose_A >= EXIT_LOSE_ROUNDS and not s.exited_A:
                    s.exited_A = True
                    s.cooldown_A = COOLDOWN_ROUNDS
                    print(f"  ⚠ {sid} 连续{EXIT_LOSE_ROUNDS}轮在A亏损，退出平台A，冷却{COOLDOWN_ROUNDS}轮")
            else:
                commA = costA = infA = 0
                profitA = 0
                marginA = 0
                salesA = 0
                gmvA = 0
                adA = 0

            # === 平台B ===
            gmvB = seller_gmv_B[sid]
            salesB = seller_sales_B[sid]
            adB = act["ad_B"]

            if not s.exited_B and sid in [x.agent_id for x in sellers_on_B]:
                commB = gmvB * polB["commission"]
                costB = s.unit_cost * salesB
                infB = seller_inf_cost_B[sid]
                profitB = gmvB - costB - commB - adB - infB
                marginB = profitB / gmvB if gmvB > 0 else (-adB if adB > 0 else 0)
                s.last_margin_B = marginB
                s.last_sales_B = salesB

                if gmvB == 0 and adB > 0:
                    s.consecutive_lose_B += 1
                elif gmvB > 0 and marginB < s.margin_safe:
                    s.consecutive_lose_B += 1
                elif gmvB > 0:
                    s.consecutive_lose_B = 0

                if s.consecutive_lose_B >= EXIT_LOSE_ROUNDS and not s.exited_B:
                    s.exited_B = True
                    s.cooldown_B = COOLDOWN_ROUNDS
                    print(f"  ⚠ {sid} 连续{EXIT_LOSE_ROUNDS}轮在B亏损，退出平台B，冷却{COOLDOWN_ROUNDS}轮")
            else:
                commB = costB = infB = 0
                profitB = 0
                marginB = 0
                salesB = 0
                gmvB = 0
                adB = 0

            seller_records.append({
                "exp": exp_label, "round": rnd,
                "seller_id": sid, "seller_type": s.seller_type, "character": s.character,
                "unit_cost": s.unit_cost, "price": act["price"],
                "ad_A": act["ad_A"], "ad_B": act["ad_B"],
                "exposure_A": exposure_A.get(sid, "无"), "exposure_B": exposure_B.get(sid, "无"),
                # 平台A
                "sales_A": salesA, "gmv_A": round(gmvA, 2),
                "comm_cost_A": round(commA, 2),
                "prod_cost_A": round(costA, 2),
                "inf_cost_A": round(infA, 2),
                "inf_driven_gmv_A": round(seller_inf_driven_gmv_A[sid], 2),
                "ad_cost_A": round(adA, 2),
                "profit_A": round(profitA, 2), "margin_A": round(marginA, 4),
                # 平台B
                "sales_B": salesB, "gmv_B": round(gmvB, 2),
                "comm_cost_B": round(commB, 2),
                "prod_cost_B": round(costB, 2),
                "inf_cost_B": round(infB, 2),
                "inf_driven_gmv_B": round(seller_inf_driven_gmv_B[sid], 2),
                "ad_cost_B": round(adB, 2),
                "profit_B": round(profitB, 2), "margin_B": round(marginB, 4),
                # 状态
                "consecutive_lose_A": s.consecutive_lose_A, "exited_A": s.exited_A, "cooldown_A": s.cooldown_A,
                "consecutive_lose_B": s.consecutive_lose_B, "exited_B": s.exited_B, "cooldown_B": s.cooldown_B,
                "thinking": act["thinking"]
            })

        # ---- 轮次汇总 ----
        total_gmv_A = sum(seller_gmv_A.values())
        total_gmv_B = sum(seller_gmv_B.values())

        # ★ 平台A营收 = 佣金收入 + 广告收入（所有广告费）- 补贴支出（方案A）
        plat_comm_A = sum(seller_gmv_A[s.agent_id] * polA["commission"] for s in sellers)
        plat_sub_A = sum(seller_gmv_A[s.agent_id] * polA["subsidy"] for s in sellers)
        plat_ad_A = sum(seller_actions[s.agent_id]["ad_A"] for s in sellers_on_A)
        plat_rev_A = plat_comm_A - plat_sub_A + plat_ad_A

        # ★ 平台B营收
        plat_comm_B = sum(seller_gmv_B[s.agent_id] * polB["commission"] for s in sellers)
        plat_sub_B = sum(seller_gmv_B[s.agent_id] * polB["subsidy"] for s in sellers)
        plat_ad_B = sum(seller_actions[s.agent_id]["ad_B"] for s in sellers_on_B)
        plat_rev_B = plat_comm_B - plat_sub_B + plat_ad_B

        inf_on_A = sum(1 for inf in influencers if inf.chosen_platform == "A")
        inf_on_B = sum(1 for inf in influencers if inf.chosen_platform == "B")
        inf_driven_tx = sum(1 for tx in round_transactions if tx["is_influencer_driven"])

        round_records.append({
            "exp": exp_label, "round": rnd,
            "A_comm": polA["commission"], "A_sub": polA["subsidy"],
            "B_comm": polB["commission"], "B_sub": polB["subsidy"],
            "gmv_A": round(total_gmv_A, 2), "gmv_B": round(total_gmv_B, 2),
            "transactions": len(round_transactions),
            "inf_driven_transactions": inf_driven_tx,
            "active_sellers_A": len(sellers_on_A), "active_sellers_B": len(sellers_on_B),
            "exited_sellers_A": sum(1 for s in sellers if s.exited_A),
            "exited_sellers_B": sum(1 for s in sellers if s.exited_B),
            "plat_comm_income_A": round(plat_comm_A, 2), "plat_sub_expense_A": round(plat_sub_A, 2),
            "plat_ad_income_A": round(plat_ad_A, 2), "plat_revenue_A": round(plat_rev_A, 2),
            "plat_comm_income_B": round(plat_comm_B, 2), "plat_sub_expense_B": round(plat_sub_B, 2),
            "plat_ad_income_B": round(plat_ad_B, 2), "plat_revenue_B": round(plat_rev_B, 2),
            "influencers_on_A": inf_on_A, "influencers_on_B": inf_on_B,
            "users_purchased": sum(1 for ud in user_round_data if ud.get("purchased") == 1)
        })

        influencer_records.extend(inf_round_data)
        user_records.extend(user_round_data)
        transaction_records.extend(round_transactions)

        print(f"  A: GMV={total_gmv_A:.0f} 营收={plat_rev_A:.0f} | B: GMV={total_gmv_B:.0f} 营收={plat_rev_B:.0f} | 交易={len(round_transactions)} 达人带货={inf_driven_tx}")

    return round_records, seller_records, influencer_records, user_records, transaction_records


# ================= 9. 运行所有16组实验 =================
all_round = []
all_seller = []
all_influencer = []
all_user = []
all_transaction = []

for idx, combo in enumerate(STRATEGY_COMBOS):
    label = f"AC{combo['A_comm']:.2f}_AS{combo['A_sub']:.2f}_BC{combo['B_comm']:.2f}_BS{combo['B_sub']:.2f}"
    polA = {"commission": combo["A_comm"], "subsidy": combo["A_sub"]}
    polB = {"commission": combo["B_comm"], "subsidy": combo["B_sub"]}
    rr, sr, ir, ur, tr = run_dual_static_experiment(polA, polB, label)
    all_round.extend(rr)
    all_seller.extend(sr)
    all_influencer.extend(ir)
    all_user.extend(ur)
    all_transaction.extend(tr)

    # 每组保存
    pd.DataFrame(all_round).to_csv(f"{RESULTS_DIR}/exp2_round_summary.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_seller).to_csv(f"{RESULTS_DIR}/exp2_seller_detail.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_influencer).to_csv(f"{RESULTS_DIR}/exp2_influencer_detail.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_user).to_csv(f"{RESULTS_DIR}/exp2_user_detail.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(all_transaction).to_csv(f"{RESULTS_DIR}/exp2_transactions.csv", index=False, encoding="utf-8-sig")

    print(f"\n✅ 实验组 {label} ({idx+1}/16) 完成")

print(f"\n{'='*60}")
print(f"✅ 实验二全部完成！共16组实验，数据保存在 {RESULTS_DIR}")
print(f"{'='*60}")


Mounted at /content/drive
加载模型: Qwen/Qwen2.5-14B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (3 outputs left out)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

模型加载完成

实验组 AC0.06_AS0.01_BC0.12_BS0.08: A佣0.06补0.01 | B佣0.12补0.08

--- AC0.06_AS0.01_BC0.12_BS0.08 第1轮 ---
  Seller_L1: 价格=47.0, 广告A=1.0, 广告B=2.0
  Seller_L2: 价格=50.0, 广告A=1.0, 广告B=3
  Seller_L3: 价格=47.0, 广告A=2.0, 广告B=1.0
  Seller_L4: 价格=50.0, 广告A=0, 广告B=3
  Seller_L5: 价格=50.0, 广告A=1.0, 广告B=2.0
  Seller_L6: 价格=49.0, 广告A=3, 广告B=2.0
  Seller_M1: 价格=150.0, 广告A=10.0, 广告B=5.0
  Seller_M2: 价格=150.0, 广告A=10.0, 广告B=5.0
  Seller_M3: 价格=150.0, 广告A=10.0, 广告B=5.0
  Seller_H1: 价格=320.0, 广告A=35.0, 广告B=10.0
    ⛔ User_M6 试图在平台A购买 Seller_H1(￥316.8)超出预算￥200，已拦截
  A: GMV=1801 营收=163 | B: GMV=1211 营收=86 | 交易=21 达人带货=9

--- AC0.06_AS0.01_BC0.12_BS0.08 第2轮 ---
  Seller_L1: 价格=47.0, 广告A=2.0, 广告B=1.0
  Seller_L2: 价格=50.0, 广告A=2.0, 广告B=3
  Seller_L3: 价格=46.0, 广告A=3, 广告B=1.0
  Seller_L4: 价格=47.0, 广告A=1.0, 广告B=2.0
  Seller_L5: 价格=50.0, 广告A=2.0, 广告B=1.0
  Seller_L6: 价格=49.0, 广告A=3, 广告B=2.0
  Seller_M1: 价格=150.0, 广告A=12.0, 广告B=3.0
  Seller_M2: 价格=160.0, 广告A=12.0, 广告B=8.0
  Seller_M3: 价格=140.0, 广告A=12.0, 广告B=3.0
